In [ ]:
# Importamos las librerías necesarias (ajustado para conda)
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from gensim import corpora, models
from gensim.models import CoherenceModel
import logging

# Configurar logging de gensim para ver el progreso
logging.basicConfig(format='%(asctime)s : %(levelname)s : %(message)s', level=logging.INFO)

# Descargar stopwords y wordnet si no están disponibles (solo la primera vez)
# Nota: La primera vez que ejecutes esto, se descargarán los datos.
nltk.download('stopwords')  
nltk.download('wordnet')

print("Todas las librerías importadas correctamente.")

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/alejandro_a/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     /home/alejandro_a/nltk_data...


Todas las librerías importadas correctamente.


In [2]:
# Cargar el dataset
conversaciones = pd.read_parquet("../data/dataset_conversaciones/dataset_50k_anonymized.parquet")
print("Dataset cargado:", conversaciones.shape)

Dataset cargado: (49999, 6)


In [3]:
# Reconstruir conversaciones
conv_full = (conversaciones
    .sort_values(["conv_id", "date"])
    .groupby("conv_id")
    .agg({
        "user_id": "first",
        "input": lambda x: " ".join(x.astype(str)),
        "channel_source": "first"
    })
    .reset_index()
    .rename(columns={"input": "texto_usuario"}))

print(f"Conversaciones reconstruidas: {len(conv_full):,}")

Conversaciones reconstruidas: 24,119


In [4]:
# Configurar stopwords
stop_words = set(stopwords.words('spanish'))
custom_stopwords = ["tarjeta", "tarjetas", "crédito", "credito", "cuenta", "cuentas",
    "puedo", "quiero", "tengo", "necesito", "saber", "puede", "favor",
    "hola", "buenos", "días", "buenas", "tardes", "gracias", "si", "es",
    "hey", "banco", "banregio", "ayuda", "ayudar", "información",
    "cliente", "clientes", "servicio", "hacer", "tener", "ser", "estar", "ir", "ver", "dar"]
stop_words = stop_words.union(custom_stopwords)

lemmatizer = WordNetLemmatizer()

def preprocess_text_gensim(text):
    text = str(text).lower()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-záéíóúñü ]", "", text)
    tokens = text.split()
    tokens = [token for token in tokens if token not in stop_words and len(token) > 2]
    tokens = [lemmatizer.lemmatize(token) for token in tokens]
    return tokens

print("Preprocesando textos...")
conv_full['tokens'] = conv_full['texto_usuario'].apply(preprocess_text_gensim)
conv_full = conv_full[conv_full['tokens'].map(len) > 0]
print(f"Conversaciones tras preprocesamiento: {len(conv_full):,}")

Preprocesando textos...
Conversaciones tras preprocesamiento: 23,976


In [5]:
dictionary = corpora.Dictionary(conv_full['tokens'])
dictionary.filter_extremes(no_below=5, no_above=0.7)
print(f"Tamaño del diccionario: {len(dictionary)}")
corpus = [dictionary.doc2bow(tokens) for tokens in conv_full['tokens']]

2026-04-26 10:30:19,157 : INFO : adding document #0 to Dictionary<0 unique tokens: []>
2026-04-26 10:30:19,288 : INFO : adding document #10000 to Dictionary<6329 unique tokens: ['automóvil', 'cuál', 'interés', 'tasa', 'aclaracion']...>
2026-04-26 10:30:19,359 : INFO : adding document #20000 to Dictionary<9004 unique tokens: ['automóvil', 'cuál', 'interés', 'tasa', 'aclaracion']...>
2026-04-26 10:30:19,388 : INFO : built Dictionary<10024 unique tokens: ['automóvil', 'cuál', 'interés', 'tasa', 'aclaracion']...> from 23976 documents (total 150091 corpus positions)
2026-04-26 10:30:19,390 : INFO : Dictionary lifecycle event {'msg': "built Dictionary<10024 unique tokens: ['automóvil', 'cuál', 'interés', 'tasa', 'aclaracion']...> from 23976 documents (total 150091 corpus positions)", 'datetime': '2026-04-26T10:30:19.389967', 'gensim': '4.4.0', 'python': '3.11.15 | packaged by conda-forge | (main, Mar  5 2026, 16:45:40) [GCC 14.3.0]', 'platform': 'Linux-6.6.87.2-microsoft-standard-WSL2-x86_64

Tamaño del diccionario: 2449


In [6]:
NUM_TOPICS = 10
PASSES = 10

print(f"Entrenando modelo LDA con {NUM_TOPICS} tópicos...")
lda_model = models.LdaModel(
    corpus=corpus, id2word=dictionary, num_topics=NUM_TOPICS,
    passes=PASSES, random_state=42, alpha='auto', eta='auto',
    chunksize=len(corpus))

print("¡Modelo LDA entrenado!")

2026-04-26 10:30:19,499 : INFO : using autotuned alpha, starting with [np.float32(0.1), np.float32(0.1), np.float32(0.1), np.float32(0.1), np.float32(0.1), np.float32(0.1), np.float32(0.1), np.float32(0.1), np.float32(0.1), np.float32(0.1)]
2026-04-26 10:30:19,501 : INFO : using serial LDA version on this node
2026-04-26 10:30:19,508 : INFO : running online (multi-pass) LDA training, 10 topics, 10 passes over the supplied corpus of 23976 documents, updating model once every 23976 documents, evaluating perplexity every 23976 documents, iterating 50x with a convergence threshold of 0.001000


Entrenando modelo LDA con 10 tópicos...


2026-04-26 10:30:23,752 : INFO : -9.078 per-word bound, 540.6 perplexity estimate based on a held-out corpus of 23976 documents with 138570 words
2026-04-26 10:30:23,753 : INFO : PROGRESS: pass 0, at document #23976/23976
2026-04-26 10:30:27,179 : INFO : optimized alpha [np.float32(0.08628494), np.float32(0.08716882), np.float32(0.08846874), np.float32(0.09196655), np.float32(0.092022), np.float32(0.08655235), np.float32(0.09221545), np.float32(0.08869174), np.float32(0.08855184), np.float32(0.091072686)]
2026-04-26 10:30:27,187 : INFO : topic #0 (0.086): 0.013*"token" + 0.011*"pago" + 0.011*"apple" + 0.010*"compra" + 0.010*"transferencia" + 0.010*"app" + 0.010*"dinero" + 0.008*"pay" + 0.008*"dice" + 0.008*"hago"
2026-04-26 10:30:27,189 : INFO : topic #5 (0.087): 0.023*"cómo" + 0.017*"dinero" + 0.015*"hago" + 0.014*"número" + 0.013*"préstamo" + 0.012*"personal" + 0.011*"transferencia" + 0.011*"veo" + 0.011*"nueva" + 0.010*"cambiar"
2026-04-26 10:30:27,190 : INFO : topic #3 (0.092): 0.0

¡Modelo LDA entrenado!


In [7]:
# NPMI (u_mass)
coherence_npmi = CoherenceModel(model=lda_model, texts=conv_full['tokens'].tolist(),
                               dictionary=dictionary, coherence='u_mass')
npmi_score = coherence_npmi.get_coherence()
print(f"Puntaje NPMI (u_mass): {npmi_score:.4f}")

# Diversidad de Tópicos (c_v)
coherence_cv = CoherenceModel(model=lda_model, texts=conv_full['tokens'].tolist(),
                              dictionary=dictionary, coherence='c_v')
topic_diversity_score = coherence_cv.get_coherence()
print(f"Puntaje C_V (Diversidad): {topic_diversity_score:.4f}")

2026-04-26 10:31:24,043 : INFO : CorpusAccumulator accumulated stats from 1000 documents
2026-04-26 10:31:24,046 : INFO : CorpusAccumulator accumulated stats from 2000 documents
2026-04-26 10:31:24,049 : INFO : CorpusAccumulator accumulated stats from 3000 documents
2026-04-26 10:31:24,052 : INFO : CorpusAccumulator accumulated stats from 4000 documents
2026-04-26 10:31:24,055 : INFO : CorpusAccumulator accumulated stats from 5000 documents
2026-04-26 10:31:24,058 : INFO : CorpusAccumulator accumulated stats from 6000 documents
2026-04-26 10:31:24,061 : INFO : CorpusAccumulator accumulated stats from 7000 documents
2026-04-26 10:31:24,064 : INFO : CorpusAccumulator accumulated stats from 8000 documents
2026-04-26 10:31:24,067 : INFO : CorpusAccumulator accumulated stats from 9000 documents
2026-04-26 10:31:24,071 : INFO : CorpusAccumulator accumulated stats from 10000 documents
2026-04-26 10:31:24,075 : INFO : CorpusAccumulator accumulated stats from 11000 documents
2026-04-26 10:31:24

Puntaje NPMI (u_mass): -5.6109


2026-04-26 10:31:24,673 : INFO : serializing accumulator to return to master...
2026-04-26 10:31:24,674 : INFO : serializing accumulator to return to master...
2026-04-26 10:31:24,673 : INFO : serializing accumulator to return to master...
2026-04-26 10:31:24,673 : INFO : serializing accumulator to return to master...
2026-04-26 10:31:24,673 : INFO : serializing accumulator to return to master...
2026-04-26 10:31:24,673 : INFO : serializing accumulator to return to master...
2026-04-26 10:31:24,674 : INFO : serializing accumulator to return to master...
2026-04-26 10:31:24,676 : INFO : serializing accumulator to return to master...
2026-04-26 10:31:24,675 : INFO : serializing accumulator to return to master...
2026-04-26 10:31:24,674 : INFO : serializing accumulator to return to master...
2026-04-26 10:31:24,680 : INFO : serializing accumulator to return to master...
2026-04-26 10:31:24,684 : INFO : accumulator serialized
2026-04-26 10:31:24,685 : INFO : accumulator serialized
2026-04-

Puntaje C_V (Diversidad): 0.3810


In [8]:
def get_dominant_topic(conversation_bow):
    topics = lda_model.get_document_topics(conversation_bow)
    if topics:
        return sorted(topics, key=lambda x: x[1], reverse=True)[0][0]
    return -1

conv_full['topic'] = [get_dominant_topic(doc) for doc in corpus]
print(conv_full['topic'].value_counts().sort_index())

topic
0    1744
1    1796
2    2967
3    3112
4    2686
5    2182
6    2208
7    2422
8    2433
9    2426
Name: count, dtype: int64


In [9]:
print("\nPalabras clave de los Tópicos:")
for idx, topic in lda_model.print_topics(num_words=10):
    print(f"Tópico {idx}: {topic}")

2026-04-26 10:31:27,886 : INFO : topic #0 (0.069): 0.040*"apple" + 0.031*"pay" + 0.024*"wallet" + 0.023*"cómo" + 0.018*"verificar" + 0.016*"fan" + 0.014*"shop" + 0.014*"reposición" + 0.013*"retiro" + 0.012*"app"
2026-04-26 10:31:27,889 : INFO : topic #1 (0.067): 0.040*"efectivo" + 0.024*"aparece" + 0.024*"dinero" + 0.022*"retirar" + 0.019*"cajero" + 0.016*"app" + 0.015*"sacar" + 0.014*"depositar" + 0.013*"aplicación" + 0.013*"comisión"
2026-04-26 10:31:27,891 : INFO : topic #2 (0.080): 0.073*"meses" + 0.053*"auto" + 0.045*"cancelar" + 0.030*"enganche" + 0.022*"mil" + 0.020*"peso" + 0.018*"automotriz" + 0.016*"personal" + 0.015*"norte" + 0.014*"pal"
2026-04-26 10:31:27,893 : INFO : topic #3 (0.081): 0.073*"cargo" + 0.066*"transferencia" + 0.025*"pago" + 0.024*"aclaración" + 0.021*"reconocido" + 0.021*"hice" + 0.020*"reconozco" + 0.019*"aparece" + 0.014*"pendiente" + 0.014*"opción"
2026-04-26 10:31:27,894 : INFO : topic #4 (0.081): 0.035*"saldo" + 0.028*"pagar" + 0.025*"recibir" + 0.024*


Palabras clave de los Tópicos:
Tópico 0: 0.040*"apple" + 0.031*"pay" + 0.024*"wallet" + 0.023*"cómo" + 0.018*"verificar" + 0.016*"fan" + 0.014*"shop" + 0.014*"reposición" + 0.013*"retiro" + 0.012*"app"
Tópico 1: 0.040*"efectivo" + 0.024*"aparece" + 0.024*"dinero" + 0.022*"retirar" + 0.019*"cajero" + 0.016*"app" + 0.015*"sacar" + 0.014*"depositar" + 0.013*"aplicación" + 0.013*"comisión"
Tópico 2: 0.073*"meses" + 0.053*"auto" + 0.045*"cancelar" + 0.030*"enganche" + 0.022*"mil" + 0.020*"peso" + 0.018*"automotriz" + 0.016*"personal" + 0.015*"norte" + 0.014*"pal"
Tópico 3: 0.073*"cargo" + 0.066*"transferencia" + 0.025*"pago" + 0.024*"aclaración" + 0.021*"reconocido" + 0.021*"hice" + 0.020*"reconozco" + 0.019*"aparece" + 0.014*"pendiente" + 0.014*"opción"
Tópico 4: 0.035*"saldo" + 0.028*"pagar" + 0.025*"recibir" + 0.024*"límite" + 0.022*"cuánto" + 0.019*"pago" + 0.018*"cuanto" + 0.016*"deuda" + 0.016*"anualidad" + 0.016*"cómo"
Tópico 5: 0.039*"cómo" + 0.030*"solicitar" + 0.029*"física" + 0.